# 06 - Walk-forward validation and signal selection

**Why not random K-fold?** It trains on the future and puts near-identical neighbouring hours on both sides of the split, so it overstates skill.

**Design:**
- expanding training window;
- 6-month validation blocks from 2019H1 to 2023H2;
- purging of training rows whose label overlaps the block.

The threshold k and the long/short vs long/flat choice are selected here, on validation only, by mean net Sharpe across folds at base costs, among configurations trading at least 100 times a year. **The test period (2024 onwards) is not touched.**

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
import run_pipeline as rp
from src import config
T, F = config.TABLES_DIR, config.FIGURES_DIR
SYMBOL = config.SYMBOL
# False (default): display the outputs saved by scripts/run_pipeline.py.
# True re-computes the stage. Do NOT re-run 'validation' or 'test' casually:
# validation resets the experiment log and every test run adds a TEST row.
RUN_STAGE = False
def show(name, **kw):
    display(Markdown(f"**{name}**")); display(pd.read_csv(T / name, **kw))
def fig(name):
    display(Image(filename=str(F / name)))
def results(section):
    return json.loads((config.REPORTS_DIR / "results.json").read_text()).get(section, {})

In [2]:
show('validation_folds.csv')

**validation_folds.csv**

,fold,train,eval,n_train,n_eval
0,2019H1,2017-08-17 -> 2018-12-31,2019-01-01 -> 2019-06-30,12043,4344
1,2019H2,2017-08-17 -> 2019-06-30,2019-07-01 -> 2019-12-31,16387,4416
2,2020H1,2017-08-17 -> 2019-12-31,2020-01-01 -> 2020-06-30,20803,4368
3,2020H2,2017-08-17 -> 2020-06-30,2020-07-01 -> 2020-12-31,25171,4416
4,2021H1,2017-08-17 -> 2020-12-31,2021-01-01 -> 2021-06-30,29587,4344
5,2021H2,2017-08-17 -> 2021-06-30,2021-07-01 -> 2021-12-31,33931,4416
6,2022H1,2017-08-17 -> 2021-12-31,2022-01-01 -> 2022-06-30,38347,4344
7,2022H2,2017-08-17 -> 2022-06-30,2022-07-01 -> 2022-12-31,42691,4416
8,2023H1,2017-08-17 -> 2022-12-31,2023-01-01 -> 2023-06-30,47107,4344
9,2023H2,2017-08-17 -> 2023-06-30,2023-07-01 -> 2023-12-31,51451,4416


In [3]:
show('validation_signal_grid.csv')

**validation_signal_grid.csv**

,k,mode,mean_fold_net_sharpe,share_folds_positive,val_sharpe,val_gross_sharpe,val_annualized_return,val_max_drawdown,val_exposure,val_trades_per_year,val_win_rate,val_cost_sum
0,0.00,long_short,-7.968503,0.0,-7.004482,0.959220,-0.993746,-1.000000,1.000000,3924.737267,0.473638,27.4456
1,0.00,long_flat,-4.561435,0.0,-3.920692,1.296698,-0.889766,-0.999984,0.548704,3919.936610,0.515028,13.7256
2,0.25,long_short,-8.232065,0.0,-7.182420,1.201345,-0.991806,-1.000000,0.849015,4939.276150,0.480060,26.8002
3,0.25,long_flat,-4.747401,0.0,-4.053611,1.515083,-0.882022,-0.999978,0.473431,3944.539978,0.515277,13.8110
4,0.50,long_short,-8.065509,0.0,-7.070985,1.344775,-0.987322,-1.000000,0.702177,5363.934283,0.488995,24.9074
5,0.50,long_flat,-4.793691,0.0,-4.119483,1.577560,-0.871431,-0.999966,0.401889,3825.723713,0.517423,13.3952
6,1.00,long_short,-7.361402,0.0,-6.554332,1.306640,-0.953814,-1.000000,0.405574,4638.234940,0.508462,17.8024
7,1.00,long_flat,-4.785246,0.0,-4.112142,1.556563,-0.794803,-0.999640,0.245146,2980.207968,0.520718,10.4356
8,1.50,long_short,-5.815579,0.0,-5.400434,0.584646,-0.792494,-0.999618,0.162847,2377.525465,0.524328,8.4784
9,1.50,long_flat,-4.255357,0.0,-3.882571,0.702168,-0.603350,-0.990323,0.103052,1511.406900,0.526598,5.2920


**Interpretation:** see the matching section of `reports/final_report.md`, written from these outputs.